# Apache Iceberg Hands-on Tutorial

This notebook demonstrates how Apache Iceberg brings version control, schema
evolution, and transactional consistency to an evolving analytical dataset.

We take the perspective of a data engineer responsible for maintaining a
longitudinal dataset of annual GDP figures published by the World Bank. Each
year a new file arrives. Sometimes it adds a year, sometimes it corrects
historical values, and sometimes it changes the structure of the data itself.

### How to read this notebook

Cells fall into two kinds:

- **❄️ Iceberg Operation** — a boxed callout precedes the cell. It states what we
  are doing, what Iceberg does underneath, and what to look for in the output.
  These are the cells that matter for the learning objectives.
- ***Data preparation step*** — ordinary pandas work. The details live in
  `gdp_pipeline.py` so that they do not compete for attention here. Read that
  file if you are curious; you do not need it to follow the tutorial.

### The two tables we maintain

| Table | Contents | Why version it |
|---|---|---|
| `gdp.raw_data.gdp_data` | the source file, stored exactly as received | detect upstream changes from the provider |
| `gdp.processed_data.gdp_data` | cleaned and imputed | track our own analytical decisions |

## Setup

In [ ]:
import pandas as pd
import pyarrow as pa

from pyiceberg.catalog import load_catalog
from pyiceberg.schema import Schema
from pyiceberg.types import NestedField, StringType, LongType, DoubleType
from pyiceberg.partitioning import PartitionSpec, PartitionField
from pyiceberg.expressions import And, In, EqualTo
from pyiceberg.exceptions import NoSuchTableError, NamespaceAlreadyExistsError

# Data-processing helpers. Everything in here is plain pandas - see gdp_pipeline.py.
from gdp_pipeline import (
    load_gdp_csv,
    clean_gdp,
    infer_iceberg_schema,
    iceberg_type_for,
    compare_schemas,
    show_warehouse,
    find_snapshot,
    reset_tutorial,
)

DATA_DIR = "/tmp/warehouse"

---
### ❄️ Iceberg Operation — CONNECT TO THE CATALOG

**What we are doing:** opening a connection to the Iceberg REST catalog running in our Docker container.

**What Iceberg does underneath:** the catalog is the entry point to everything. It holds one pointer per table, and that pointer names the table's current metadata file. It stores no data itself.

**Watch for in the output:** an empty list of namespaces on a clean run. Nothing has been created yet.

**Corresponds to:** Figure 1 in the paper.

In [ ]:
catalog = load_catalog("default", **{"uri": "http://iceberg-rest-catalog:8181"})

print("Namespaces in the catalog:", catalog.list_namespaces())

> **Lost your place?** Uncomment and run the cell below to drop every table and
> namespace this notebook creates, then start again from the top. The expected
> snapshot counts quoted later only hold on a clean run.

In [ ]:
# reset_tutorial(catalog)

## Year 2021 — the first ingestion

The first file arrives, covering 1960 through 2021. There is no table yet, so
this section creates one and loads the data into it.

### Raw data

*Data preparation step — ordinary pandas work, no Iceberg involved.*

Read the source CSV into a DataFrame. No transformation is applied — we will store this exactly as the provider sent it.

In [ ]:
df_raw_2021 = load_gdp_csv(f"{DATA_DIR}/gdp_data_until_2021.csv")

df_raw_2021.head()

---
### ❄️ Iceberg Operation — CREATE TABLE

**What we are doing:** creating an empty Iceberg table with an explicit, typed schema derived from the DataFrame.

**What Iceberg does underneath:** unlike a CSV, an Iceberg table has a real schema in which every column carries a stable *field ID*. Those IDs are what let Iceberg rename or drop a column later without touching a single data file.

**Watch for in the output:** the field IDs printed next to each column. Note also that the table is created empty — creating a table and writing data to it are separate operations.

In [ ]:
raw_schema = infer_iceberg_schema(df_raw_2021)

RAW_TABLE = "gdp.raw_data.gdp_data"

try:
    catalog.create_namespace("gdp.raw_data")
except NamespaceAlreadyExistsError:
    pass

try:
    catalog.drop_table(RAW_TABLE)
    print(f"\nDropped pre-existing table {RAW_TABLE}")
except NoSuchTableError:
    pass

raw_table = catalog.create_table(RAW_TABLE, raw_schema)
raw_table

---
### ❄️ Iceberg Operation — APPEND

**What we are doing:** writing the 1960-2021 data into the empty table, tagged with metadata describing where it came from.

**What Iceberg does underneath:** Iceberg writes a Parquet file, creates a metadata file containing snapshot 1, and advances the catalog pointer to it. The three steps commit atomically.

**Watch for in the output:** a snapshot with operation `append`. The `snapshot_properties` we attach here are not decoration: later in this notebook we use them to *find* this exact version of the data again.

**Corresponds to:** Figure 2 in the paper.

In [ ]:
properties = {
    "job-name": "initial_gdp_load",
    "source-file": "gdp_data_until_2021.csv",
    "comment": "Full load from World Bank CSV, data until 2021",
}

raw_table.append(pa.Table.from_pandas(df_raw_2021), snapshot_properties=properties)

for snapshot in raw_table.snapshots():
    print(snapshot.snapshot_id, "|", snapshot.summary.get("operation"), "|",
          snapshot.summary.get("comment"))

Now look at what is actually on disk. This view is the single most useful thing
in the notebook: it makes the metadata layer visible instead of theoretical.

In [ ]:
show_warehouse(raw_table, label="after the first append")

---
### ❄️ Iceberg Operation — READ

**What we are doing:** reading the table back to confirm what was stored.

**What Iceberg does underneath:** the query engine asks the catalog for the current metadata file, reads which data files that snapshot references, and scans only those.

**Watch for in the output:** the row and column counts match the CSV we loaded.

In [ ]:
df_check = raw_table.scan().to_pandas()

print("shape:", df_check.shape, " year range:", df_check["date"].min(), "-", df_check["date"].max())
df_check.head()

### Processed data

*Data preparation step — ordinary pandas work, no Iceberg involved.*


`clean_gdp()` performs four steps, all of them documented in `gdp_pipeline.py`:

1. drops aggregate rows with no `countryiso3code` (these are regions and income
   groups such as "Euro area" and "World", not countries);
2. drops countries with no GDP observation in any year;
3. imputes the remaining gaps using the strategy we choose;
4. sorts and resets the index.

None of this is Iceberg-specific. It is ordinary pipeline work that any project
would do. We isolate it here so the rest of the notebook can focus on what
Iceberg does with the result.

In [ ]:
df_processed_2021 = clean_gdp(df_raw_2021, impute="ffill")

df_processed_2021.head()

---
### ❄️ Iceberg Operation — CREATE TABLE + APPEND

**What we are doing:** storing the cleaned data in a *second* Iceberg table, separate from the raw one.

**What Iceberg does underneath:** two independent tables means two independent histories. The raw table's history records what the provider sent us; the processed table's history records the decisions we made. Neither can overwrite the other.

**Watch for in the output:** a second table appearing in its own namespace, with its own snapshot 1.

In [ ]:
PROCESSED_TABLE = "gdp.processed_data.gdp_data"

try:
    catalog.create_namespace("gdp.processed_data")
except NamespaceAlreadyExistsError:
    pass

try:
    catalog.drop_table(PROCESSED_TABLE)
    print(f"Dropped pre-existing table {PROCESSED_TABLE}")
except NoSuchTableError:
    pass

processed_table = catalog.create_table(
    PROCESSED_TABLE,
    infer_iceberg_schema(df_processed_2021, verbose=False),
)

processed_table.append(
    pa.Table.from_pandas(df_processed_2021),
    snapshot_properties={
        "job-name": "processed_gdp",
        "source-file": "gdp.raw_data.gdp_data",
        "comment": "Cleaned 2021 data, forward fill imputation",
    },
)

show_warehouse(processed_table, label="after the first processed load")

## Year 2022 — a replacement file arrives

A year later the provider publishes a new file covering 1960 through 2022. It is
a *complete* file, not an increment: it supersedes the one we already have, and
some historical values in it may have been revised.

Because the new file replaces rather than extends the old one, the correct
operation is an overwrite, not an append.

### Raw data

In [ ]:
df_raw_2022 = load_gdp_csv(f"{DATA_DIR}/gdp_data_until_2022.csv")

---
### ❄️ Iceberg Operation — SCHEMA CHECK

**What we are doing:** comparing the columns of the incoming file against the columns of the existing table.

**What Iceberg does underneath:** Iceberg enforces its schema on write. Checking first means we find out about a structural change deliberately, instead of discovering it through a failed job.

**Watch for in the output:** `Schema match.` this time. The 2023 file later in the notebook will not be so cooperative.

In [ ]:
compare_schemas(raw_table, df_raw_2022)

---
### ❄️ Iceberg Operation — OVERWRITE

**What we are doing:** replacing the entire contents of the raw table with the new file.

**What Iceberg does underneath:** Iceberg writes a new Parquet file, creates a new metadata file whose snapshot references only that file, and advances the catalog pointer. The 2021 Parquet file is **not deleted** — it is simply no longer referenced by the current snapshot.

**Watch for in the output:** **two** new snapshots, not one: a `delete` followed by an `append`. An overwrite is a delete-then-append pair committed as a single atomic unit. (Some PyIceberg versions report this as one `overwrite` snapshot instead — either way, the previous data files are retained.) This surprises almost everyone the first time.

**Corresponds to:** Figure 4 in the paper.

In [ ]:
raw_table.overwrite(
    pa.Table.from_pandas(df_raw_2022),
    snapshot_properties={
        "job-name": "gdp_refresh_2022",
        "source-file": "gdp_data_until_2022.csv",
        "comment": "Full load from World Bank CSV, data until 2022",
    },
)

for snapshot in raw_table.snapshots():
    print(snapshot.snapshot_id, "|", snapshot.summary.get("operation"), "|",
          snapshot.summary.get("comment"))

Now look at the warehouse again. This is the moment where the difference between
Iceberg and copying a file becomes concrete.

In [ ]:
show_warehouse(raw_table, label="after the 2022 overwrite")

Two Parquet files exist on disk, but the current snapshot reads only one of them.
The 2021 file was not deleted and not modified. That is precisely what makes the
time travel query later in this notebook possible, and it is why versioning in
Iceberg costs a metadata file rather than a second copy of the dataset.

### Processed data — two competing imputation strategies

How should we fill the gaps in the GDP series? Two analysts can reasonably
disagree, and the choice materially changes the numbers that reach the dashboard
and the forecasting model.

Rather than saving `processed_v1.csv` and `processed_v2.csv`, we write both into
the *same* Iceberg table. Each becomes a snapshot. The history then records not
just the data but the decision behind it.

*Data preparation step — ordinary pandas work, no Iceberg involved.*

Strategy 1 — forward fill, then backward fill, within each country.

In [ ]:
df_ffill = clean_gdp(df_raw_2022, impute="ffill")

---
### ❄️ Iceberg Operation — OVERWRITE (version 1 of 2)

**What we are doing:** replacing the processed table with the forward-filled version of the 2022 data.

**What Iceberg does underneath:** another delete-and-append pair, and another snapshot. Note the comment we attach: it names the strategy used.

**Watch for in the output:** the snapshot list growing. Nothing is lost when we overwrite again in a moment.

In [ ]:
processed_table.overwrite(
    pa.Table.from_pandas(df_ffill),
    snapshot_properties={
        "job-name": "processed_gdp",
        "source-file": "gdp.raw_data.gdp_data",
        "comment": "Cleaned 2022 data, forward fill imputation",
    },
)
print("Rows now in the processed table:", len(processed_table.scan().to_pandas()))

*Data preparation step — ordinary pandas work, no Iceberg involved.*

Strategy 2 — interpolate interior gaps and extrapolate the ends along a fitted trend line, clipping any negative result to zero.

In [ ]:
df_linear = clean_gdp(df_raw_2022, impute="linear")

---
### ❄️ Iceberg Operation — OVERWRITE (version 2 of 2)

**What we are doing:** overwriting the *same* table again, this time with the interpolated version.

**What Iceberg does underneath:** a third pair of snapshots. Both analytical versions now coexist in one table's history, distinguished by the comment attached to each commit — no duplicate files, no `_final_v2` in a filename.

**Watch for in the output:** the snapshot list. Both strategies are recoverable, and we will retrieve each of them by name in the next section.

In [ ]:
processed_table.overwrite(
    pa.Table.from_pandas(df_linear),
    snapshot_properties={
        "job-name": "processed_gdp",
        "source-file": "gdp.raw_data.gdp_data",
        "comment": "Cleaned 2022 data, linear interpolation imputation",
    },
)

for snapshot in processed_table.snapshots():
    print(snapshot.snapshot_id, "|", snapshot.summary.get("operation"), "|",
          snapshot.summary.get("comment"))

## Time travel

Every state the tables have passed through is still readable. This section
retrieves earlier versions of both tables without restoring a backup, without
taking anything offline, and without having kept a single duplicate file.

### Inspecting the history

In [ ]:
print(f"{'snapshot id':<22} {'operation':<10} comment")
print("-" * 100)
for snapshot in raw_table.snapshots():
    print(f"{snapshot.snapshot_id:<22} "
          f"{str(snapshot.summary.get('operation')):<10} "
          f"{snapshot.summary.get('comment')}")

---
### ❄️ Iceberg Operation — TIME TRAVEL

**What we are doing:** reading the raw table as it existed after the very first load, when it only went up to 2021.

**What Iceberg does underneath:** the query engine reads the current metadata file, walks its snapshot log back to the snapshot we asked for, and reads the Parquet files that snapshot referenced. The newer files are never touched.

**Watch for in the output:** the year range ends at 2021, even though the table's current state runs to 2022.

**Corresponds to:** Figure 5 in the paper.

In [ ]:
# Look the snapshot up by the comment we attached when we wrote it.
# Indexing into snapshots() by position is fragile: re-running any write cell
# shifts every index after it, and a time travel query would then silently
# return the wrong data.
snapshot_2021 = find_snapshot(raw_table, contains="until 2021")

df_2021_state = raw_table.scan(snapshot_id=snapshot_2021.snapshot_id).to_pandas()

print("\nAs of that snapshot :", df_2021_state.shape, "| years",
      df_2021_state["date"].min(), "-", df_2021_state["date"].max())

df_now = raw_table.scan().to_pandas()
print("Current state       :", df_now.shape, "| years",
      df_now["date"].min(), "-", df_now["date"].max())

### Recovering a specific analytical decision

---
### ❄️ Iceberg Operation — TIME TRAVEL

**What we are doing:** retrieving both imputation strategies from the processed table and comparing them side by side.

**What Iceberg does underneath:** the two versions live in the same table, distinguished only by snapshot. Neither was ever copied to a separate file.

**Watch for in the output:** different GDP values for the same country and year under the two strategies. This is the concrete payoff of versioning an analytical decision rather than a filename.

In [ ]:
snap_ffill = find_snapshot(processed_table, contains="2022 data, forward fill")
print()
snap_linear = find_snapshot(processed_table, contains="2022 data, linear interpolation")

version_ffill = processed_table.scan(snapshot_id=snap_ffill.snapshot_id).to_pandas()
version_linear = processed_table.scan(snapshot_id=snap_linear.snapshot_id).to_pandas()

comparison = (
    version_ffill[["country", "date", "value"]]
    .merge(
        version_linear[["country", "date", "value"]],
        on=["country", "date"],
        suffixes=("_ffill", "_linear"),
    )
)
comparison["difference"] = comparison["value_linear"] - comparison["value_ffill"]

n_diff = int((comparison["difference"].abs() > 1).sum())
print(f"\n{n_diff} of {len(comparison)} rows differ between the two strategies.")
print("Same country, same year, same source file - a different number reaching the dashboard.")

comparison[comparison["difference"].abs() > 1].head(10)

## Year 2023 — the structure of the data changes

The 2023 file arrives and the `month` column is gone. The provider changed the
shape of the data.

This is schema drift, and it is the failure mode that breaks pipelines built on
the assumption of a stable structure. Iceberg handles it as a metadata change.

### Raw data

In [ ]:
df_raw_2023 = load_gdp_csv(f"{DATA_DIR}/gdp_data_until_2023.csv")

---
### ❄️ Iceberg Operation — SCHEMA CHECK

**What we are doing:** comparing the new file against the table before attempting to write anything.

**What Iceberg does underneath:** the column sets no longer match. Writing now would fail; Iceberg will not silently accept data that does not fit the declared schema.

**Watch for in the output:** `month` reported as present in the table but missing from the new data.

In [ ]:
dropped_columns, added_columns = compare_schemas(raw_table, df_raw_2023)

---
### ❄️ Iceberg Operation — SCHEMA EVOLUTION

**What we are doing:** dropping the `month` column from the table's schema so the new file fits.

**What Iceberg does underneath:** Iceberg writes a new metadata file containing the revised schema. Because every column has a stable field ID, removing one is a bookkeeping change. **No Parquet file is read, rewritten, or deleted.**

**Watch for in the output:** the operation completing instantly regardless of table size, and — in the warehouse listing below — no new data file appearing.

**Corresponds to:** Figure 6 in the paper.

In [ ]:
if dropped_columns:
    with raw_table.update_schema() as update:
        for column in dropped_columns:
            print(f"Dropping column from the schema: {column}")
            update.delete_column(column)

if added_columns:
    with raw_table.update_schema() as update:
        for column in added_columns:
            iceberg_type = iceberg_type_for(df_raw_2023[column])
            print(f"Adding column to the schema: {column} ({iceberg_type})")
            update.add_column(column, iceberg_type)

print("\nSchema is now:")
for field in raw_table.schema().fields:
    print(f"  field_id={field.field_id:<3} {field.name:<16} {field.field_type}")

In [ ]:
show_warehouse(raw_table, label="after dropping a column")

Compare this listing with the one before the schema change. The number of Parquet
files is unchanged and none of them was rewritten. Dropping a column from a
dataset of any size cost one small JSON file.

This is the point students most often find hard to believe. The data files still
physically contain a `month` column; the schema simply no longer declares it, so
no reader returns it.

---
### ❄️ Iceberg Operation — OVERWRITE

**What we are doing:** writing the 2023 data under the revised schema.

**What Iceberg does underneath:** a delete-and-append pair, as before, now governed by the new schema.

**Watch for in the output:** the `month` column absent from the result, and the year range extending to 2023.

In [ ]:
raw_table.overwrite(
    pa.Table.from_pandas(df_raw_2023),
    snapshot_properties={
        "job-name": "gdp_refresh_2023",
        "source-file": "gdp_data_until_2023.csv",
        "comment": "Full load from World Bank CSV, data until 2023, month column dropped",
    },
)

df_now = raw_table.scan().to_pandas()
print("shape:", df_now.shape, "| columns:", list(df_now.columns))
df_now.head()

---
### ❄️ Iceberg Operation — TIME TRAVEL ACROSS A SCHEMA CHANGE

**What we are doing:** reading the table as it stood in 2021, before the column was dropped.

**What Iceberg does underneath:** each snapshot is bound to the schema in force when it was written, so Iceberg returns the historical rows with their historical structure.

**Watch for in the output:** the `month` column reappearing, and the year range ending at 2021. The table's past is intact even though its present has a different shape.

In [ ]:
snapshot_2021 = find_snapshot(raw_table, contains="until 2021")
df_2021_state = raw_table.scan(snapshot_id=snapshot_2021.snapshot_id).to_pandas()

print("\ncolumns as of 2021:", list(df_2021_state.columns))
print("columns right now  :", list(raw_table.scan().to_pandas().columns))
df_2021_state.head()

### Processed data

*Data preparation step — ordinary pandas work, no Iceberg involved.*


We keep this step deliberately simple and skip imputation for 2023 — but we still
apply the row filtering, so the processed table continues to hold only actual
countries. Skipping the filter would quietly put "Euro area" and "World" rows
into the processed table and break the distinction the two tables exist to draw.

In [ ]:
df_processed_2023 = clean_gdp(df_raw_2023, impute=None)

---
### ❄️ Iceberg Operation — SCHEMA EVOLUTION + OVERWRITE

**What we are doing:** applying the same column drop to the processed table, then loading the 2023 data.

**What Iceberg does underneath:** each table carries its own schema and its own history. The change we made to the raw table has no effect here, so we make it again, explicitly.

**Watch for in the output:** the processed table's schema losing `month` independently of the raw table's.

In [ ]:
# Recompute the difference against *this* table rather than reusing the variable
# from the raw-data section - the two tables evolve independently.
dropped_processed, _ = compare_schemas(processed_table, df_processed_2023)

if dropped_processed:
    with processed_table.update_schema() as update:
        for column in dropped_processed:
            print(f"Dropping column from the schema: {column}")
            update.delete_column(column)

processed_table.overwrite(
    pa.Table.from_pandas(df_processed_2023),
    snapshot_properties={
        "job-name": "processed_gdp",
        "source-file": "gdp.raw_data.gdp_data",
        "comment": "Cleaned 2023 data, no imputation, month column dropped",
    },
)

print("\nProcessed table columns:", [f.name for f in processed_table.schema().fields])

## Partitioning

Suppose the dashboard almost always queries data from 2000 onward. Scanning every
year on every query wastes work.

Iceberg has no conditional partitions, so we add a column that categorises each
row as `before_2000` or `since_2000` and partition on that. Note this is an
*explicit* partition column, chosen here for clarity; Iceberg also supports
hidden partitioning, where the partition value is derived from an existing
column by a transform such as `year(date)`.

*Data preparation step — ordinary pandas work, no Iceberg involved.*

Load the current processed data and add the partition column.

In [ ]:
df_to_partition = processed_table.scan().to_pandas()

df_to_partition["year_partition"] = df_to_partition["date"].apply(
    lambda year: "before_2000" if year < 2000 else "since_2000"
)

df_to_partition["year_partition"].value_counts()

---
### ❄️ Iceberg Operation — CREATE PARTITIONED TABLE

**What we are doing:** creating a new table that is physically organised by the partition column.

**What Iceberg does underneath:** Iceberg writes one set of data files per partition value and records in the metadata which files belong to which partition. That mapping is what lets a query skip files without opening them.

**Watch for in the output:** the partition spec attached to the table. We derive the partition column's field ID from the schema rather than hard-coding it, so this cell keeps working if the columns ever change again.

**Corresponds to:** Figure 7 in the paper.

In [ ]:
PARTITIONED_TABLE = "gdp.processed_data.gdp_data_partitioned"

partitioned_schema = infer_iceberg_schema(df_to_partition, verbose=False)

# Look the field ID up by name instead of hard-coding a number.
partition_field_id = partitioned_schema.find_field("year_partition").field_id
print(f"Partitioning on 'year_partition' (field_id={partition_field_id})")

spec = PartitionSpec(
    PartitionField(
        source_id=partition_field_id,
        field_id=1000,
        name="year_partition",
        transform="identity",
    )
)

try:
    catalog.drop_table(PARTITIONED_TABLE)
    print(f"Dropped pre-existing table {PARTITIONED_TABLE}")
except NoSuchTableError:
    pass

partitioned_table = catalog.create_table(
    PARTITIONED_TABLE, schema=partitioned_schema, partition_spec=spec
)

partitioned_table.append(
    pa.Table.from_pandas(df_to_partition),
    snapshot_properties={
        "job-name": "partitioned_gdp",
        "source-file": "gdp.processed_data.gdp_data",
        "comment": "Processed 2023 data partitioned by year_partition",
    },
)

show_warehouse(partitioned_table, label="one data file per partition")

---
### ❄️ Iceberg Operation — PARTITION PRUNING

**What we are doing:** asking Iceberg to *plan* three scans, and reporting how much data each plan would actually have to read.

**What Iceberg does underneath:** Iceberg evaluates the filter against partition metadata **before opening any data file**. Files that cannot contain matching rows are excluded from the plan.

**Watch for in the output:** the record counts. On a dataset this small the wall-clock difference is invisible, which is why we measure the plan rather than time the query. The planned record count is the mechanism; elapsed time is just a consequence of it.

In [ ]:
def scan_cost(scan):
    # How much work would this scan do? Ask the plan, do not time the query.
    tasks = list(scan.plan_files())
    return len(tasks), sum(task.file.record_count for task in tasks)

plans = {
    "partitioned, no filter      ": partitioned_table.scan(),
    "partitioned, since_2000     ": partitioned_table.scan(
        row_filter="year_partition == 'since_2000'"),
    "unpartitioned, date >= 2000 ": processed_table.scan(row_filter="date >= 2000"),
}

print(f"{'scan':<30} {'files':>7} {'records read':>14}")
print("-" * 53)
for label, scan in plans.items():
    n_files, n_records = scan_cost(scan)
    print(f"{label:<30} {n_files:>7} {n_records:>14,}")

print()
print("The partitioned filter skips a whole file. The unpartitioned filter cannot:")
print("its single data file holds every year, so every record must be read and")
print("then discarded. Same answer, different amount of work.")

> **Note.** We could have partitioned the existing processed table instead of
> creating a new one. That would require adding the column to its schema, applying
> a new partition spec, and rewriting the existing data so that the older rows are
> organised under the new layout too. Iceberg supports evolving a partition spec
> without rewriting history, in which case old and new data simply live under
> different specs.

## UPSERT — correcting individual records

Four countries have published corrected GDP figures for 2023. We receive a small
file containing only those rows.

Updating a few records means deleting the old rows and inserting the new ones.
Those two operations must either both succeed or both fail: a reader must never
observe a state in which the old rows are gone and the new ones have not yet
arrived.

In [ ]:
df_updates = load_gdp_csv(f"{DATA_DIR}/gdp_updates_2023.csv")

df_updates

---
### ❄️ Iceberg Operation — ATOMIC TRANSACTION (delete + append)

**What we are doing:** deleting the 2023 rows for the affected countries and appending the corrected rows, as one indivisible unit.

**What Iceberg does underneath:** both operations are staged in a single transaction and committed as one snapshot. If either fails, the catalog pointer never moves and the table is untouched.

**Watch for in the output:** a success message and a single new snapshot covering both operations — not two separate commits.

In [ ]:
countries_to_update = tuple(df_updates["country"].unique())
print("Correcting 2023 values for:", countries_to_update)

with processed_table.transaction() as txn:
    txn.delete(And(EqualTo("date", 2023), In("country", countries_to_update)))
    txn.append(
        pa.Table.from_pandas(df_updates),
        snapshot_properties={
            "job-name": "gdp_corrections_2023",
            "source-file": "gdp_updates_2023.csv",
            "comment": "Upserted corrected 2023 GDP values for four countries",
        },
    )

print("\nUpsert committed.")

In [ ]:
df_after = processed_table.scan().to_pandas()

df_after[(df_after["date"] == 2023) & (df_after["country"].isin(countries_to_update))]

---
### ❄️ Iceberg Operation — TIME TRAVEL

**What we are doing:** comparing the affected rows immediately before and immediately after the correction.

**What Iceberg does underneath:** the pre-correction snapshot is still fully readable, which is what makes the change auditable.

**Watch for in the output:** the original values before the upsert and the corrected values after it, side by side.

In [ ]:
snap_before = find_snapshot(processed_table, contains="no imputation")

df_before = processed_table.scan(snapshot_id=snap_before.snapshot_id).to_pandas()

mask_before = (df_before["date"] == 2023) & (df_before["country"].isin(countries_to_update))
mask_after = (df_after["date"] == 2023) & (df_after["country"].isin(countries_to_update))

audit = (
    df_before.loc[mask_before, ["country", "date", "value"]]
    .merge(
        df_after.loc[mask_after, ["country", "date", "value"]],
        on=["country", "date"],
        suffixes=("_before", "_after"),
    )
)
audit

## What Iceberg did

| Operation | Data files written | Data files rewritten | Cost |
|---|---|---|---|
| Create table | 0 | 0 | one metadata file |
| Append | 1 | 0 | one metadata file |
| Overwrite | 1 | 0 | one metadata file, old files retained |
| Drop a column | **0** | **0** | one metadata file |
| Partition | one per partition | 0 | one metadata file |
| Upsert | 1 | 0 | one metadata file |
| Time travel | 0 | 0 | nothing — it is a read |

Every version of this dataset that we produced remains queryable. At no point did
we duplicate the dataset, name a file `_final_v2`, take anything offline, or lose
the ability to answer the question "what did this look like before?"

### Recap of the learning objectives

- **create, update, and query Iceberg tables** — create, append, overwrite, scan
- **snapshot-based versioning with custom metadata** — `snapshot_properties`, and
  using them to locate a specific version later
- **revert datasets to earlier states using time travel** — `scan(snapshot_id=...)`,
  including across a schema change
- **manage schema evolution** — `update_schema()` as a metadata-only operation